In [6]:
import site
cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(cutlass_pkg_path)

import os, shutil, stat

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/kaggle/working/ptxas-blackwell"

shutil.copy2(src, dst)
os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = dst


In [7]:
!echo $TRITON_PTXAS_BLACKWELL_PATH

/kaggle/working/ptxas-blackwell


In [ ]:
!pip install -q --no-index --find-links /kaggle/input/trl-offline/trl_packages trl
!pip install -q --no-index --find-links /kaggle/input/wandb-offline/wandb_packages wandb


In [ ]:
import sys
sys.path.insert(0, "/kaggle/input/nemotron-rl-approach")

from nemotron_grpo import (
    GRPOExperimentConfig,
    run_experiment,
    load_csv_dataset,
    load_model_and_tokenizer,
    apply_lora,
    build_grpo_trainer,
)


In [ ]:
config = GRPOExperimentConfig(
    model_path="/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    train_csv="/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv",
    output_dir="/kaggle/working/grpo_run",
    # ── Hyperparameters ──────────────────────────────────────────
    learning_rate=3e-6,
    per_device_train_batch_size=16,
    num_generations=4,
    max_completion_length=7680,
    temperature=1.0,
    epsilon=10.0,
    max_steps=10,
    logging_steps=1,
    # ── Optional extras (uncomment to override TRL defaults) ─────
    # warmup_steps=30,
    # weight_decay=0.1,
    # lr_scheduler_type="cosine",
    # optim="adamw_8bit",
    # max_grad_norm=0.1,
    # beta=0.0,
    # gradient_checkpointing=True,
    # ── Rewards ──────────────────────────────────────────────────
    reward_functions=["cosine_reward", "format_reward", "length_reward"],
    # ── W&B (offline by default, no internet needed) ─────────────
    wandb_project="nemotron-grpo",
    wandb_run_name="baseline",
    wandb_tags=["kaggle", "baseline"],
    wandb_mode="offline",
    wandb_dir="/kaggle/working/wandb",
)

model = run_experiment(config)


In [ ]:
import subprocess

subprocess.run("zip -m submission.zip *", shell=True, check=True)


In [ ]:
print('Done.')
